In [1]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.7 MB/s eta 0:00:00


In [16]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [8]:
!pip install -q langchain-huggingface
from langchain_huggingface.llms import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace

In [47]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-Coder-Next",
    task='text-generation'
)

model=ChatHuggingFace(llm=llm)


In [ ]:
import os
hf_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")


1) tool creation

In [48]:
@tool
def multiply(a:int , b:int)->int:
  """Given 2 numbers a and b , this tool return multiplication of those 2 numbers"""
  return a*b

In [49]:
print(multiply.invoke({'a':2 , 'b' : 3}))

6


In [50]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b , this tool return multiplication of those 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


2) tool - binding

In [51]:
llm_with_tools=model.bind_tools([multiply])

3) tool - calling

In [52]:
llm_with_tools.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 315, 'total_tokens': 325}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c8f4c-0854-7611-bfd0-ab5216e50745-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 315, 'output_tokens': 10, 'total_tokens': 325})

In [53]:
llm_with_tools.invoke("Can you mutliply 3 with 10")

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 10}', 'name': 'multiply', 'description': None}, 'id': 'call_4113932bc7594fed977b32f5', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 325, 'total_tokens': 356}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8f4c-1131-7010-bbb6-22a8ce6d7f99-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_4113932bc7594fed977b32f5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 325, 'output_tokens': 31, 'total_tokens': 356})

In [67]:
query=HumanMessage(content="Can you mutliply 3 with 1000")

In [68]:
messages=[query]

In [69]:
messages

[HumanMessage(content='Can you mutliply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [70]:
result=llm_with_tools.invoke(messages)

In [71]:
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_83ed13cd78774491a0639984', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 327, 'total_tokens': 360}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8f51-00dd-7b30-9bff-9d67e4d62c77-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_83ed13cd78774491a0639984', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 327, 'output_tokens': 33, 'total_tokens': 360})

In [72]:
messages.append(result)

In [73]:
messages

[HumanMessage(content='Can you mutliply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_83ed13cd78774491a0639984', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 327, 'total_tokens': 360}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8f51-00dd-7b30-9bff-9d67e4d62c77-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_83ed13cd78774491a0639984', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 327, 'output_tokens': 33, 'total_tokens': 360})]

In [74]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 1000},
 'id': 'call_83ed13cd78774491a0639984',
 'type': 'tool_call'}

4) Tool - Execution

In [75]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='3000', name='multiply', tool_call_id='call_83ed13cd78774491a0639984')

we get a tool message by this which can be sent to llm to get a proper respond

In [76]:
tool_result=multiply.invoke(result.tool_calls[0])

In [83]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_83ed13cd78774491a0639984')

In [77]:
messages.append(tool_result)
messages

[HumanMessage(content='Can you mutliply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_83ed13cd78774491a0639984', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 327, 'total_tokens': 360}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c8f51-00dd-7b30-9bff-9d67e4d62c77-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_83ed13cd78774491a0639984', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 327, 'output_tokens': 33, 'total_tokens': 360}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_83ed13cd78774491a0639984')]

In [78]:
llm_with_tools.invoke(messages)

AIMessage(content='The result of multiplying 3 by 1000 is 3000.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 377, 'total_tokens': 397}, 'model_name': 'Qwen/Qwen3-Coder-Next', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c8f51-3961-7490-874c-2c8ef4c3a226-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 377, 'output_tokens': 20, 'total_tokens': 397})

In [79]:
llm_with_tools.invoke(messages).content

'The result of multiplying 3 by 1000 is **3000**.'